# Fusion de Datasets
### Dataset 2018 (volpatto, ~1311 filas) + Dataset 2023 (fatihb, 207 filas)

---

## Objetivo
Combinar ambos datasets de la misma fuente (Coffee Quality Institute) para obtener un dataset unificado de ~1500 filas, solucionando el problema de muestra pequena que limita la fiabilidad estadistica del modelo.

## Desafios a resolver
| Problema | Dataset | Solucion |
|----------|---------|----------|
| Nombres de columnas con puntos | volpatto 2018 | Renombrar a espacios |
| Columna indice `Unnamed: 0` | Ambos | Eliminar |
| Columnas con nombres distintos | Ambos | Renombrar para alinear |
| Columnas exclusivas de un dataset | Ambos | Conservar solo las comunes |

---

## Flujo del Notebook
1. Cargar ambos datasets
2. Inspeccionar diferencias
3. Normalizar columnas (renombrar, alinear)
4. Seleccionar columnas comunes
5. Fusionar y validar
6. Guardar el CSV unificado

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Librerias importadas')

---
## BLOQUE 1 -- Cargar ambos datasets

In [ ]:
# --- Cargar dataset 2018 (volpatto) ---
df_2018 = pd.read_csv('../../data/raw/arabica_data_cleaned.csv')

# --- Cargar dataset 2023 (fatihb) ---
df_2023 = pd.read_csv('../../data/raw/coffee_quality.csv')

print(f'Dataset 2018 (volpatto): {df_2018.shape[0]} filas x {df_2018.shape[1]} columnas')
print(f'Dataset 2023 (fatihb):   {df_2023.shape[0]} filas x {df_2023.shape[1]} columnas')

---
## BLOQUE 2 -- Inspeccionar diferencias entre datasets

Antes de fusionar, necesitamos entender que columnas tiene cada uno
y como se llaman. Los dos vienen de la misma fuente (CQI) pero fueron scrapeados
en anos distintos y por personas distintas, por lo que los nombres no coinciden exactamente.

In [ ]:
print('--- Columnas Dataset 2018 (volpatto) ---')
for i, c in enumerate(df_2018.columns, 1):
    print(f'  {i:02d}. {c}')

print()
print('--- Columnas Dataset 2023 (fatihb) ---')
for i, c in enumerate(df_2023.columns, 1):
    print(f'  {i:02d}. {c}')

---
## BLOQUE 3 -- Normalizar columnas

El dataset 2018 usa **puntos** como separador en los nombres: `Country.of.Origin`.  
El dataset 2023 usa **espacios**: `Country of Origin`.  

> Solucion: estandarizamos el 2018 al formato del 2023.

In [ ]:
# --- Paso 1: Reemplazar puntos por espacios en los nombres de columna del 2018 ---
df_2018.columns = df_2018.columns.str.replace('.', ' ', regex=False)

# --- Paso 2: Eliminar columnas indice inutiles ---
if 'Unnamed: 0' in df_2018.columns:
    df_2018 = df_2018.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in df_2023.columns:
    df_2023 = df_2023.drop(columns=['Unnamed: 0'])
if 'ID' in df_2023.columns:
    df_2023 = df_2023.drop(columns=['ID'])

# --- Paso 3: Renombrar columnas del 2018 para que coincidan con el 2023 ---
renombrar_2018 = {
    'Moisture'          : 'Moisture Percentage',
    'In Country Partner' : 'In-Country Partner',
    'Cupper Points'     : 'Overall',
}
df_2018 = df_2018.rename(columns=renombrar_2018)

print(f'Dataset 2018 normalizado: {df_2018.shape[1]} columnas')
print(f'Dataset 2023 normalizado: {df_2023.shape[1]} columnas')

In [ ]:
# --- Verificar que columnas son exclusivas de cada dataset ---
solo_2018 = sorted(set(df_2018.columns) - set(df_2023.columns))
solo_2023 = sorted(set(df_2023.columns) - set(df_2018.columns))

print(f'Solo en 2018: {solo_2018}')
print(f'Solo en 2023: {solo_2023}')

---
## BLOQUE 4 -- Seleccionar columnas comunes

Solo nos quedamos con las columnas que existen en AMBOS datasets.
Esto garantiza que el dataset fusionado sea consistente.

In [ ]:
cols_comunes = sorted(set(df_2018.columns) & set(df_2023.columns))

print(f'Columnas comunes ({len(cols_comunes)}):')
for c in cols_comunes:
    print(f'  - {c}')

# Recortar cada dataset a las columnas comunes
df_2018_final = df_2018[cols_comunes].copy()
df_2023_final = df_2023[cols_comunes].copy()

print(f'\n2018 recortado: {df_2018_final.shape}')
print(f'2023 recortado: {df_2023_final.shape}')

---
## BLOQUE 5 -- Fusionar los datasets

Apilamos las filas de ambos datasets uno debajo del otro con `pd.concat`,
ya que las columnas estan alineadas.

In [ ]:
df_fusion = pd.concat([df_2018_final, df_2023_final], ignore_index=True)

print(f'Dataset fusionado: {df_fusion.shape[0]} filas x {df_fusion.shape[1]} columnas')

---
## BLOQUE 6 -- Validar el dataset final

In [ ]:
print('===========================================')
print('    VALIDACION DEL DATASET FUSIONADO')
print('===========================================')
print(f'  Filas totales:   {df_fusion.shape[0]}')
print(f'  Columnas:        {df_fusion.shape[1]}')

# Nulos por columna
nulos = df_fusion.isnull().sum()
nulos_pct = (nulos / len(df_fusion) * 100).round(2)
print(f'\n  Porcentaje global de nulos: {(df_fusion.isnull().sum().sum() / df_fusion.size * 100):.2f}%')

print('\n  Nulos por columna:')
for col in df_fusion.columns:
    if nulos[col] > 0:
        print(f'    {col:<30} {nulos[col]:>5} ({nulos_pct[col]}%)')

print('===========================================')

In [ ]:
df_fusion.head()

In [ ]:
df_fusion.info()

---
## BLOQUE 7 -- Guardar el dataset fusionado

In [ ]:
import os
os.makedirs('../../data/processed/Camila', exist_ok=True)

output_path = '../../data/processed/Camila/coffee_quality_fusion.csv'
df_fusion.to_csv(output_path, index=False)

print(f'Dataset fusionado guardado en: {output_path}')
print(f'Dimensiones finales: {df_fusion.shape[0]} filas x {df_fusion.shape[1]} columnas')
print()
print('===========================================')
print('   RESUMEN DE LA FUSION')
print('===========================================')
print(f'  Dataset 2023 original:   207 filas')
print(f'  Dataset 2018 anadido:    1311 filas')
print(f'  Total final:             {len(df_fusion)} filas')